In [1]:
# =============================================================================
# Dataset 2: mind-wandering classification - Ablation Study
# Evaluating contribution of (1) Feature Families, (2) Scalp Regions, and (3) Frequency Bands
# =============================================================================

import os
from pathlib import Path
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold

warnings.filterwarnings("ignore")

# ============================================================
# PATHS & CONFIGURATION
# ============================================================
FEATURE_CSV = Path("/kaggle/input/datasets/jvkrishwanth/d2-ablation/all_epoch_features.csv")
RAW_BAND_CSV = Path("/kaggle/input/datasets/jvkrishwanth/d2-ablation/raw_wide_features.csv")
CLEAN_BAND_CSV = Path("/kaggle/input/datasets/jvkrishwanth/d2-ablation/ica_cleaned_wide_features.csv")

OUTDIR = Path("/kaggle/working/")
PLOT_DIR = Path("/kaggle/working/plots/")

OUTDIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
CONDITION_COLORS = {"Raw": "#66c2a5", "ICA_Cleaned": "#fc8d62"}
BANDS = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def make_models():
    """Return SVM and Logistic Regression pipelines."""
    return {
        "SVM": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
        ]),
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")),
        ]),
    }

def make_group_cv(groups, max_splits=5):
    n_splits = min(max_splits, len(np.unique(groups)))
    if n_splits < 2:
        raise ValueError("At least two groups are required for grouped cross-validation.")
    try:
        return StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    except Exception:
        return GroupKFold(n_splits=n_splits)

def decision_scores(model, features):
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(features), dtype=float)
    return np.asarray(model.predict_proba(features)[:, 1], dtype=float)

def map_features(df):
    """Group feature columns by functional family and spatial scalp region (64 channels)."""
    metadata = {
        "Subject", "Session", "Subject_Session", "Window", "ArtifactCondition",
        "Epoch_Index", "Epoch_Sample", "State", "Label",
    }
    feature_cols = [col for col in df.columns if col not in metadata]
    
    by_family = {
        "Time-Domain": [],
        "Hjorth": [],
        "Spectral": [],
        "Entropy": []
    }
    by_region = {
        "Anterior (Frontal)": [],
        "Central & Temporal": [],
        "Posterior": []
    }
    
    family_keywords = {
        "Time-Domain": ["Mean", "Variance", "Std", "RMS", "Kurtosis", "Skewness", "Peak2Peak", "Energy"],
        "Hjorth": ["HjorthActivity", "HjorthMobility", "HjorthComplexity"],
        "Spectral": ["TotalPower", "ThetaPower", "AlphaPower", "DominantFreq"],
        "Entropy": ["SpectralEntropy", "ShannonEntropy", "DifferentialEntropy"]
    }
    
    for col in feature_cols:
        parts = col.split("__")
        if len(parts) != 2:
            continue
        channel, feature = parts
        
        # 1. Map to functional family
        matched_family = None
        for fam, keywords in family_keywords.items():
            if feature in keywords:
                by_family[fam].append(col)
                matched_family = fam
                break
        
        # 2. Map to scalp region (64 channels montage)
        channel_upper = channel.upper()
        if channel_upper.startswith("FT"):
            by_region["Central & Temporal"].append(col)
        elif channel_upper.startswith(("FP", "AF", "F", "FC")):
            by_region["Anterior (Frontal)"].append(col)
        elif channel_upper.startswith(("C", "T", "CP", "TP")):
            by_region["Central & Temporal"].append(col)
        elif channel_upper.startswith(("P", "O", "I", "PO")):
            by_region["Posterior"].append(col)
            
    return feature_cols, by_family, by_region

def map_bands(df):
    """Group bandwise feature columns by frequency band."""
    metadata = {"Condition", "Subject", "Session", "Subject_Session", "Epoch", "Window", "State", "Label", "ArtifactCondition"}
    feature_cols = [col for col in df.columns if col not in metadata]
    
    by_band = {b: [] for b in BANDS}
    for col in feature_cols:
        col_lower = col.lower()
        for b in BANDS:
            if b.lower() in col_lower:
                by_band[b].append(col)
                break
    return feature_cols, by_band

def evaluate_subset(df, cols, group_column):
    """Run cross-validated evaluation on a subset of columns."""
    X = df[cols]
    y = df["Label"].to_numpy(dtype=int)
    groups = df[group_column].to_numpy()
    
    if len(np.unique(y)) < 2:
        raise ValueError("Both classes are required for classification.")
        
    cv = make_group_cv(groups)
    results = []
    
    for model_name, template in make_models().items():
        oof_prediction = np.full(len(y), -1, dtype=int)
        oof_score = np.full(len(y), np.nan, dtype=float)
        
        for train_index, test_index in cv.split(X, y, groups):
            if len(np.unique(y[train_index])) < 2:
                continue
            model = clone(template)
            model.fit(X.iloc[train_index], y[train_index])
            oof_prediction[test_index] = model.predict(X.iloc[test_index])
            oof_score[test_index] = decision_scores(model, X.iloc[test_index])
            
        valid = oof_prediction >= 0
        if not valid.any():
            continue
        y_true, y_pred, y_score = y[valid], oof_prediction[valid], oof_score[valid]
        
        results.append({
            "Model": model_name,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, y_score),
        })
        
    return results

# ============================================================
# MAIN ABLATION STUDY LOOP
# ============================================================

def main():
    print("=" * 80)
    print("DATASET 2 - INTEGRATED ABLATION STUDY (FAMILIES, REGIONS, BANDS)")
    print("=" * 80)
    
    # --------------------------------------------------------
    # PART 1: FEATURE FAMILY & SCALP REGION ABLATION
    # --------------------------------------------------------
    if not FEATURE_CSV.exists():
        raise FileNotFoundError(f"Feature CSV not found at: {FEATURE_CSV}")
        
    print(f"Loading feature family/region features from: {FEATURE_CSV}")
    df_fam = pd.read_csv(FEATURE_CSV)
    
    feature_cols, by_family, by_region = map_features(df_fam)
    print(f"Features parsed: {len(feature_cols)}")
    
    windows = df_fam["Window"].unique()
    conditions = df_fam["ArtifactCondition"].unique()
    
    all_ablation_results = []
    
    # Define configurations
    ablation_configs = [
        ("Baseline", "Full Feature Set", lambda: feature_cols)
    ]
    for fam in by_family.keys():
        cols_to_keep = [c for c in feature_cols if c not in by_family[fam]]
        ablation_configs.append(("Feature Family (Omit)", f"Omit {fam}", lambda c=cols_to_keep: c))
    for fam, cols in by_family.items():
        ablation_configs.append(("Feature Family (Isolate)", f"Only {fam}", lambda c=cols: c))
    for reg in by_region.keys():
        cols_to_keep = [c for c in feature_cols if c not in by_region[reg]]
        ablation_configs.append(("Scalp Region (Omit)", f"Omit {reg}", lambda c=cols_to_keep: c))
    for reg, cols in by_region.items():
        ablation_configs.append(("Scalp Region (Isolate)", f"Only {reg}", lambda c=cols: c))
        
    # Run Part 1
    for window in windows:
        for condition in conditions:
            print(f"\nEvaluating Window: {window} | Condition: {condition}")
            subset = df_fam[(df_fam["Window"] == window) & (df_fam["ArtifactCondition"] == condition)].copy()
            
            for category, variant, col_selector in ablation_configs:
                cols = col_selector()
                if len(cols) == 0:
                    continue
                
                print(f"  Running: {category} - {variant} ({len(cols)} features)...")
                try:
                    res = evaluate_subset(subset, cols, "Subject")
                    for r in res:
                        r.update({
                            "Window": window,
                            "ArtifactCondition": condition,
                            "Category": category,
                            "Variant": variant,
                            "N_Features": len(cols)
                        })
                        all_ablation_results.append(r)
                except Exception as e:
                    print(f"    Error running {variant}: {e}")
                    
    # --------------------------------------------------------
    # PART 2: FREQUENCY BAND ABLATION (Delta, Theta, Alpha, Beta, Gamma)
    # --------------------------------------------------------
    if not RAW_BAND_CSV.exists() or not CLEAN_BAND_CSV.exists():
        print(f"\n[Warning] Bandwise feature files not found. Skipping band ablation.")
    else:
        print(f"\nLoading raw bandwise features from: {RAW_BAND_CSV}")
        df_raw = pd.read_csv(RAW_BAND_CSV)
        df_raw["ArtifactCondition"] = "Raw"
        
        print(f"Loading cleaned bandwise features from: {CLEAN_BAND_CSV}")
        df_clean = pd.read_csv(CLEAN_BAND_CSV)
        df_clean["ArtifactCondition"] = "ICA_Cleaned"
        
        df_band = pd.concat([df_raw, df_clean], ignore_index=True)
        del df_raw, df_clean
        gc.collect()
        
        band_features, by_band = map_bands(df_band)
        print(f"Bandwise features parsed: {len(band_features)}")
        for b, cols in by_band.items():
            print(f"  Band '{b}': {len(cols)} columns")
            
        band_configs = [
            ("Baseline (Band)", "Full Band Feature Set", lambda: band_features)
        ]
        for b in BANDS:
            cols_to_keep = [c for c in band_features if c not in by_band[b]]
            band_configs.append(("Frequency Band (Omit)", f"Omit {b}", lambda c=cols_to_keep: c))
        for b in BANDS:
            band_configs.append(("Frequency Band (Isolate)", f"Only {b}", lambda b_name=b: by_band[b_name]))
            
        # Run Part 2
        for condition in df_band["ArtifactCondition"].unique():
            print(f"\nEvaluating Band Ablation | Condition: {condition}")
            subset = df_band[df_band["ArtifactCondition"] == condition].copy()
            
            for category, variant, col_selector in band_configs:
                cols = col_selector()
                if len(cols) == 0:
                    continue
                
                print(f"  Running: {category} - {variant} ({len(cols)} features)...")
                try:
                    res = evaluate_subset(subset, cols, "Subject")
                    for r in res:
                        r.update({
                            "Window": "Analysis_Window",
                            "ArtifactCondition": condition,
                            "Category": category,
                            "Variant": variant,
                            "N_Features": len(cols)
                        })
                        all_ablation_results.append(r)
                except Exception as e:
                    print(f"    Error running {variant}: {e}")
                    
    # Save combined results
    results_df = pd.DataFrame(all_ablation_results)
    results_df.to_csv(OUTDIR / "ablation_metrics.csv", index=False)
    print(f"\nSaved all metrics to: {OUTDIR / 'ablation_metrics.csv'}")
    
    # ============================================================
    # PLOTTING COMPARISONS
    # ============================================================
    sns.set_theme(style="whitegrid", context="notebook")
    
    for model in ["SVM", "Logistic Regression"]:
        for metric in ["Accuracy", "F1", "ROC-AUC"]:
            model_df = results_df[results_df["Model"] == model]
            if model_df.empty:
                continue
                
            # 1. Feature Family Omission Plot
            fig, ax = plt.subplots(figsize=(10, 6))
            plot_data = model_df[model_df["Category"].isin(["Baseline", "Feature Family (Omit)"])]
            if not plot_data.empty:
                sns.barplot(data=plot_data, x="Variant", y=metric, hue="ArtifactCondition", palette=CONDITION_COLORS, ax=ax)
                ax.set_ylim(0.4, 1.0)
                ax.set_title(f"{model} - Feature Family Omission Study ({metric})")
                ax.set_xlabel("Configuration", fontweight='bold')
                ax.set_ylabel(metric, fontweight='bold')
                plt.setp(ax.get_xticklabels(), fontweight='bold')
                plt.setp(ax.get_yticklabels(), fontweight='bold')
                ax.legend(title="Pre-processing")
                for container in ax.containers:
                    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9, fontweight='bold')
                plt.tight_layout()
                plt.savefig(PLOT_DIR / f"{model.lower().replace(' ', '_')}_feature_omission_{metric.lower()}.png", dpi=250)
            plt.close()
            
            # 2. Scalp Region Omission Plot
            fig, ax = plt.subplots(figsize=(10, 6))
            plot_data_reg = model_df[model_df["Category"].isin(["Baseline", "Scalp Region (Omit)"])]
            if not plot_data_reg.empty:
                sns.barplot(data=plot_data_reg, x="Variant", y=metric, hue="ArtifactCondition", palette=CONDITION_COLORS, ax=ax)
                ax.set_ylim(0.4, 1.0)
                ax.set_title(f"{model} - Scalp Region Omission Study ({metric})")
                ax.set_xlabel("Configuration", fontweight='bold')
                ax.set_ylabel(metric, fontweight='bold')
                plt.setp(ax.get_xticklabels(), fontweight='bold')
                plt.setp(ax.get_yticklabels(), fontweight='bold')
                ax.legend(title="Pre-processing")
                for container in ax.containers:
                    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9, fontweight='bold')
                plt.tight_layout()
                plt.savefig(PLOT_DIR / f"{model.lower().replace(' ', '_')}_region_omission_{metric.lower()}.png", dpi=250)
            plt.close()
            
            # 3. Frequency Band Omission Plot
            plot_data_band = model_df[model_df["Category"].isin(["Baseline (Band)", "Frequency Band (Omit)"])].copy()
            if not plot_data_band.empty:
                fig, ax = plt.subplots(figsize=(10, 6))
                # Sort bands order for consistent plotting (use Category to identify baseline row)
                def band_sort_key(row):
                    if row["Category"] == "Baseline (Band)":
                        return -1
                    band = row["Variant"].replace("Omit ", "")
                    return BANDS.index(band) if band in BANDS else 99
                plot_data_band["SortOrder"] = plot_data_band.apply(band_sort_key, axis=1)
                plot_data_band = plot_data_band.sort_values("SortOrder")
                
                sns.barplot(data=plot_data_band, x="Variant", y=metric, hue="ArtifactCondition", palette=CONDITION_COLORS, ax=ax)
                ax.set_ylim(0.4, 1.0)
                ax.set_title(f"{model} - Frequency Band Omission Study ({metric})")
                ax.set_xlabel("Configuration", fontweight='bold')
                ax.set_ylabel(metric, fontweight='bold')
                plt.setp(ax.get_xticklabels(), fontweight='bold')
                plt.setp(ax.get_yticklabels(), fontweight='bold')
                ax.legend(title="Pre-processing")
                for container in ax.containers:
                    ax.bar_label(container, fmt="%.3f", padding=2, fontsize=9, fontweight='bold')
                plt.tight_layout()
                plt.savefig(PLOT_DIR / f"{model.lower().replace(' ', '_')}_band_omission_{metric.lower()}.png", dpi=250)
                plt.close()

    print(f"Saved all plots to: {PLOT_DIR}")
    print("Dataset 2 ablation study complete!")

if __name__ == "__main__":
    main()


DATASET 2 - INTEGRATED ABLATION STUDY (FAMILIES, REGIONS, BANDS)
Loading feature family/region features from: /kaggle/input/datasets/jvkrishwanth/d2-ablation/all_epoch_features.csv
Features parsed: 1152

Evaluating Window: Analysis_Window | Condition: Raw
  Running: Baseline - Full Feature Set (1152 features)...
  Running: Feature Family (Omit) - Omit Time-Domain (640 features)...
  Running: Feature Family (Omit) - Omit Hjorth (960 features)...
  Running: Feature Family (Omit) - Omit Spectral (896 features)...
  Running: Feature Family (Omit) - Omit Entropy (960 features)...
  Running: Feature Family (Isolate) - Only Time-Domain (512 features)...
  Running: Feature Family (Isolate) - Only Hjorth (192 features)...
  Running: Feature Family (Isolate) - Only Spectral (256 features)...
  Running: Feature Family (Isolate) - Only Entropy (192 features)...
  Running: Scalp Region (Omit) - Omit Anterior (Frontal) (720 features)...
  Running: Scalp Region (Omit) - Omit Central & Temporal (792 f